# Notebook 2 — Data Cleaning & Preprocessing
## Split → Clean → Engineer → Save

**Purpose**: Transform raw data into model-ready datasets. All imputation statistics
are computed from **training data only** to prevent data leakage.

**Outputs**: Cleaned CSVs in `data/processed/` + feature metadata JSON.

**Key principle**: We save cleaned but **NOT encoded/scaled** data — the final
sklearn Pipeline (Notebook 3) handles encoding and scaling internally so the
serialized model can accept raw feature values at inference time.

In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
RAW_PATH = Path("../data/raw/AmesHousing.csv")
PROCESSED_DIR = Path("../data/processed")

## 1) Load Raw Data and Normalize Column Names

The ISU Ames CSV uses space-separated column names (e.g., "Overall Qual"). We strip
spaces and slashes to match the CamelCase Kaggle convention used in the bootcamp.

In [ ]:
# Load and normalize column names
df = pd.read_csv(RAW_PATH)
df.columns = df.columns.str.replace(" ", "").str.replace("/", "")

# Drop database identifiers (no predictive value)
df = df.drop(columns=["Order", "PID"])

assert "OverallQual" in df.columns, "Column rename failed"
assert "SalePrice" in df.columns, "Target column missing"
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Target: SalePrice (mean=${df['SalePrice'].mean():,.0f}, median=${df['SalePrice'].median():,.0f})")
df.head(3)

## 2) Train / Validation / Test Split — 60/20/20 (BEFORE Cleaning)

**Critical**: We split **before any cleaning** to prevent data leakage. All imputation
statistics (medians, modes) will be computed from `X_train` only, then applied to
val and test via `.transform()`.

| Split | Size | Role | Rule |
|-------|------|------|------|
| Train | 60% | Learn patterns | Fit everything here |
| Validation | 20% | Tune/compare | Transform only |
| Test | 20% | Final evaluation | Touch exactly once |

In [ ]:
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

# Step 1: 60% train, 40% temporary holdout
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE
)

# Step 2: split the 40% holdout evenly -> 20% val, 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

total = len(df)
print(f"Train:      {len(X_train):4d} rows  ({len(X_train)/total*100:.0f}%)")
print(f"Validation: {len(X_val):4d} rows  ({len(X_val)/total*100:.0f}%)")
print(f"Test:       {len(X_test):4d} rows  ({len(X_test)/total*100:.0f}%)")
print(f"Total:      {total:4d} rows")

# Verify no index overlap between splits
assert len(set(X_train.index) & set(X_val.index)) == 0, "Train/val overlap!"
assert len(set(X_train.index) & set(X_test.index)) == 0, "Train/test overlap!"
assert len(set(X_val.index) & set(X_test.index)) == 0, "Val/test overlap!"
print("\nNo index overlap — splits are clean.")

In [ ]:
# Split quality check: SalePrice distributions should be similar
split_comparison = pd.DataFrame({
    "Train": y_train.describe(),
    "Validation": y_val.describe(),
    "Test": y_test.describe(),
})
print("SalePrice statistics across splits:")
print(split_comparison.round(0))

# Overlay histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, y_split, color in [("Train", y_train, "blue"), ("Val", y_val, "orange"), ("Test", y_test, "green")]:
    axes[0].hist(y_split, bins=40, alpha=0.4, label=label, color=color)
    axes[1].hist(np.log1p(y_split), bins=40, alpha=0.4, label=label, color=color)

axes[0].set_title("SalePrice Distribution by Split")
axes[0].legend()
axes[1].set_title("log1p(SalePrice) by Split")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3) Duplicate Detection and Removal

Check for exact duplicate rows within each split. Duplicates can inflate training
metrics and waste computation.

In [ ]:
# Check for duplicates within each split
for name, X_split in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    n_dupes = X_split.duplicated().sum()
    print(f"{name}: {n_dupes} duplicates")
    if n_dupes > 0:
        print(f"  → Dropping {n_dupes} duplicates from {name}")

# Drop duplicates from training (keep first occurrence)
dupes_before = len(X_train)
mask = X_train.duplicated()
if mask.sum() > 0:
    X_train = X_train[~mask]
    y_train = y_train.loc[X_train.index]
    print(f"\nTrain: {dupes_before} → {len(X_train)} rows ({dupes_before - len(X_train)} removed)")
else:
    print("\nNo duplicates found — proceeding.")

## 4) Missing Value Audit (Train Set Only)

We audit nulls on the training set to understand the imputation problem. Every null
column is classified into one of four buckets based on *why* it is missing.

In [ ]:
# Null summary on training set only
null_counts = X_train.isnull().sum()
null_pct = (null_counts / len(X_train) * 100).round(2)
null_summary = pd.DataFrame({
    "null_count": null_counts,
    "null_pct": null_pct,
    "dtype": X_train.dtypes
}).query("null_count > 0").sort_values("null_pct", ascending=False)

print(f"Columns with nulls: {len(null_summary)} out of {X_train.shape[1]}")
print(f"Total null values in train: {X_train.isnull().sum().sum():,}\n")

# Split by dtype for strategy planning
print("Null columns by dtype:")
print(null_summary.groupby("dtype")["null_count"].count())
print()
null_summary

## 5) Domain-Driven Imputation Strategy

We classify every null column into one of four buckets:

| Strategy | Fill Value | Rationale |
|----------|-----------|-----------|
| Structural categorical | `"None"` | NaN = feature doesn't exist (no pool, no garage) |
| Structural numeric | `0` | NaN = no such structure, area/count is zero |
| Recording gap numeric | Train median | Genuine missing measurement (e.g., LotFrontage) |
| Recording gap categorical | Train mode | Very few nulls, recording errors |

In [ ]:
# Categorical nulls that mean "not present" -> fill with "None"
categorical_none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence",
    "FireplaceQu", "GarageType", "GarageFinish",
    "GarageQual", "GarageCond", "BsmtQual", "BsmtCond",
    "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "MasVnrType",
]

# Numeric nulls that mean "not present" -> fill with 0
numeric_zero_cols = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath", "MasVnrArea",
]

# Genuinely unknown numeric -> fill with median (fit on train)
numeric_median_cols = ["LotFrontage"]

# Genuinely unknown categorical -> fill with mode (fit on train)
categorical_mode_cols = [
    "Electrical", "MSZoning", "Utilities", "Functional",
    "KitchenQual", "Exterior1st", "Exterior2nd", "SaleType",
]

# Verify all null columns are accounted for
all_handled = (categorical_none_cols + numeric_zero_cols +
               numeric_median_cols + categorical_mode_cols)
unhandled = [c for c in null_summary.index if c not in all_handled]
print(f"Unhandled null columns: {unhandled if unhandled else 'None — all covered'}")

In [ ]:
# Apply imputation: fit on train, transform all splits
# (Same pattern as bootcamp week_1_day_4_solution.ipynb cell 39)

def apply_imputation(X_tr, X_v, X_te,
                     cat_none_cols, num_zero_cols,
                     num_median_cols, cat_mode_cols):
    """Apply domain-driven imputation. Fit statistics on train only."""
    X_tr = X_tr.copy()
    X_v = X_v.copy()
    X_te = X_te.copy()

    # Categorical structural -> "None"
    for col in cat_none_cols:
        if col in X_tr.columns:
            X_tr[col] = X_tr[col].fillna("None")
            X_v[col] = X_v[col].fillna("None")
            X_te[col] = X_te[col].fillna("None")

    # Numeric structural -> 0
    for col in num_zero_cols:
        if col in X_tr.columns:
            X_tr[col] = X_tr[col].fillna(0)
            X_v[col] = X_v[col].fillna(0)
            X_te[col] = X_te[col].fillna(0)

    # Numeric unknown -> median (fit on train only)
    med_imp = SimpleImputer(strategy="median")
    for col in num_median_cols:
        if col in X_tr.columns:
            X_tr[[col]] = med_imp.fit_transform(X_tr[[col]])
            X_v[[col]] = med_imp.transform(X_v[[col]])
            X_te[[col]] = med_imp.transform(X_te[[col]])

    # Categorical unknown -> mode (fit on train only)
    mode_imp = SimpleImputer(strategy="most_frequent")
    for col in cat_mode_cols:
        if col in X_tr.columns:
            X_tr[[col]] = mode_imp.fit_transform(X_tr[[col]])
            X_v[[col]] = mode_imp.transform(X_v[[col]])
            X_te[[col]] = mode_imp.transform(X_te[[col]])

    return X_tr, X_v, X_te


# Keep un-imputed copies for missingness indicator
X_train_raw = X_train.copy()

X_train_imp, X_val_imp, X_test_imp = apply_imputation(
    X_train, X_val, X_test,
    categorical_none_cols, numeric_zero_cols,
    numeric_median_cols, categorical_mode_cols,
)
print("Imputation applied.")

In [ ]:
# Verify zero nulls in all splits
assert X_train_imp.isnull().sum().sum() == 0, "Train still has nulls!"
assert X_val_imp.isnull().sum().sum() == 0, "Val still has nulls!"
assert X_test_imp.isnull().sum().sum() == 0, "Test still has nulls!"
print("All nulls resolved across all 3 splits.")

# Before/after histogram for LotFrontage
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
X_train_raw["LotFrontage"].hist(bins=40, ax=axes[0], alpha=0.7)
axes[0].set_title("LotFrontage BEFORE imputation")
X_train_imp["LotFrontage"].hist(bins=40, ax=axes[1], alpha=0.7)
axes[1].set_title("LotFrontage AFTER imputation (median fill)")
plt.tight_layout()
plt.show()

## 6) Missingness Indicators

For columns where the *fact* of being null carries information, we add a binary flag.
LotFrontage is the main candidate — if it weakly correlates with SalePrice, the
indicator adds marginal signal.

In [ ]:
# Add LotFrontage_was_null indicator (from original un-imputed data)
for X_orig, X_imp in [(X_train_raw, X_train_imp), (X_val, X_val_imp), (X_test, X_test_imp)]:
    X_imp["LotFrontage_was_null"] = X_orig["LotFrontage"].isnull().astype(int)

print(f"LotFrontage_was_null value counts (train):")
print(X_train_imp["LotFrontage_was_null"].value_counts())

# Check correlation with SalePrice
r = X_train_imp["LotFrontage_was_null"].corr(y_train)
print(f"\nCorrelation with SalePrice: r = {r:.3f}")
print("(Weak correlation — indicator has marginal value but is kept for completeness)")

## 7) Outlier Handling

Remove the well-documented GrLivArea outliers from **training only**. These are two
properties with >4,000 sqft sold below $200k — likely non-arm's-length transactions.
We do NOT remove them from val/test (they represent real-world data).

In [ ]:
# Remove known GrLivArea outliers from training
outlier_mask = (X_train_imp["GrLivArea"] > 4000) & (y_train < 200000)
n_outliers = outlier_mask.sum()
print(f"Removing {n_outliers} known GrLivArea outliers from training set")

X_train_imp = X_train_imp[~outlier_mask]
y_train = y_train.loc[X_train_imp.index]
print(f"Training set after outlier removal: {len(X_train_imp)} rows")

## 8) Feature Engineering

Create derived features that combine related raw features. Each is computed identically
across all three splits (no data leakage — these are deterministic transforms).

In [ ]:
def engineer_features(X):
    """Create derived features from existing columns."""
    X = X.copy()
    # Total square footage (basement + floors)
    X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
    # House age at time of sale
    X["HouseAge"] = X["YrSold"] - X["YearBuilt"]
    # Total bathrooms (full + half weighted)
    X["TotalBath"] = (X["FullBath"] + 0.5 * X["HalfBath"]
                      + X["BsmtFullBath"] + 0.5 * X["BsmtHalfBath"])
    # Binary: was the house remodeled?
    X["HasRemodel"] = (X["YearRemodAdd"] != X["YearBuilt"]).astype(int)
    return X


X_train_imp = engineer_features(X_train_imp)
X_val_imp = engineer_features(X_val_imp)
X_test_imp = engineer_features(X_test_imp)

print(f"New columns added: TotalSF, HouseAge, TotalBath, HasRemodel")
print(f"Shape after engineering: {X_train_imp.shape}")

# Quick sanity check
print(f"\nTotalSF range: {X_train_imp['TotalSF'].min():.0f} - {X_train_imp['TotalSF'].max():.0f}")
print(f"HouseAge range: {X_train_imp['HouseAge'].min()} - {X_train_imp['HouseAge'].max()}")
print(f"TotalBath range: {X_train_imp['TotalBath'].min()} - {X_train_imp['TotalBath'].max()}")

## 9) Feature Type Classification for Downstream

Classify all features by processing type. This metadata flows to Notebook 3 where
it drives the ColumnTransformer configuration.

In [ ]:
# Feature type classification (extended from bootcamp week_2_day_2 cells 7, 11)

# Ordinal features with explicit orderings
ordinal_features = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
    "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC",
    "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "GarageFinish",
    "Functional", "LandSlope", "LotShape", "PavedDrive", "Fence",
]

# Ordinal orderings (from bootcamp week_2_day_2 cell 11, extended)
quality_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ordinal_orders = {
    "ExterQual": quality_order,
    "ExterCond": quality_order,
    "BsmtQual": quality_order,
    "BsmtCond": quality_order,
    "HeatingQC": quality_order,
    "KitchenQual": quality_order,
    "FireplaceQu": quality_order,
    "GarageQual": quality_order,
    "GarageCond": quality_order,
    "PoolQC": quality_order,
    "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "GarageFinish": ["None", "Unf", "RFn", "Fin"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "PavedDrive": ["N", "P", "Y"],
    "Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"],
}

# Nominal: remaining object columns
nominal_features = [c for c in X_train_imp.select_dtypes(include="object").columns
                    if c not in ordinal_features]

# Numeric: all number columns
numeric_features = X_train_imp.select_dtypes(include="number").columns.tolist()

# Engineered features (subset of numeric)
engineered_features = ["TotalSF", "HouseAge", "TotalBath", "HasRemodel"]

print(f"Numeric:  {len(numeric_features)} features")
print(f"Ordinal:  {len(ordinal_features)} features")
print(f"Nominal:  {len(nominal_features)} features")
print(f"Engineered (in numeric): {engineered_features}")
print(f"\nNominal: {nominal_features}")

## 10) Save Processed Data

Save cleaned (imputed + engineered) data to `data/processed/`. Files contain
original types — strings for categoricals, original numeric scales. Encoding and
scaling happen inside the sklearn Pipeline (Notebook 3).

In [ ]:
# Build feature metadata for Notebook 3
feature_metadata = {
    "ordinal": ordinal_features,
    "nominal": nominal_features,
    "numeric": numeric_features,
    "engineered": engineered_features,
    "ordinal_orders": ordinal_orders,
}

# Save everything
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

X_train_imp.to_csv(PROCESSED_DIR / "X_train.csv", index=True)
X_val_imp.to_csv(PROCESSED_DIR / "X_val.csv", index=True)
X_test_imp.to_csv(PROCESSED_DIR / "X_test.csv", index=True)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=True, header=True)
y_val.to_csv(PROCESSED_DIR / "y_val.csv", index=True, header=True)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=True, header=True)

with open(PROCESSED_DIR / "feature_metadata.json", "w") as f:
    json.dump(feature_metadata, f, indent=2)

print("Saved to data/processed/:")
for p in sorted(PROCESSED_DIR.glob("*")):
    print(f"  {p.name:30s}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Reload and verify
X_train_check = pd.read_csv(PROCESSED_DIR / "X_train.csv", index_col=0)
y_train_check = pd.read_csv(PROCESSED_DIR / "y_train.csv", index_col=0).squeeze()

print("Verification after reload:")
print(f"  X_train: {X_train_check.shape}, nulls: {X_train_check.isnull().sum().sum()}")
print(f"  y_train: {y_train_check.shape}, nulls: {y_train_check.isnull().sum()}")
print(f"  Dtypes: {X_train_check.dtypes.value_counts().to_dict()}")
print(f"\n  X_val:   {pd.read_csv(PROCESSED_DIR / 'X_val.csv', index_col=0).shape}")
print(f"  X_test:  {pd.read_csv(PROCESSED_DIR / 'X_test.csv', index_col=0).shape}")

## 11) Summary

### What we built
| Step | Output |
|------|--------|
| Loaded ISU Ames CSV | 2,930 rows × 80 columns (after dropping Order, PID) |
| Renamed columns | CamelCase (OverallQual, GrLivArea, etc.) |
| Split 60/20/20 | Before any cleaning — no leakage |
| Removed duplicates | Within-split only |
| Imputed nulls | 4 strategies: structural cat/num, median, mode |
| Added missingness indicator | LotFrontage_was_null |
| Removed outliers | 2 GrLivArea anomalies from training |
| Engineered features | TotalSF, HouseAge, TotalBath, HasRemodel |
| Saved | 6 CSVs + feature_metadata.json to data/processed/ |

### What remains
- **Notebook 3**: Statistical feature selection (Pearson, Spearman, ANOVA, MI) →
  select 10 features → build sklearn Pipeline → train Ridge + GradientBoosting →
  serialize best model to `models/`